In [ ]:
import pandas as pd
import numpy as np
import neurokit2 as nk
import pickle
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')

FS = 700
WINDOW_SEC = 30
STEP_SEC = 5  # 5-second step creates 25s overlap
SAMPLES_PER_WINDOW = FS * WINDOW_SEC
SAMPLES_PER_STEP = FS * STEP_SEC

SUBJECT_IDS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
DATA_PATH = 'WESAD/' 

all_windows_data = []

print(f"Starting extraction: {WINDOW_SEC}s windows, {STEP_SEC}s steps.")

for subject in tqdm(SUBJECT_IDS, desc="Processing Subjects"):
    file_path = os.path.join(DATA_PATH, subject, f'{subject}.pkl')
    if not os.path.exists(file_path):
        continue
        
    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
        
    labels = data['label']
    ecg_raw = data['signal']['chest']['ECG'].flatten()
    eda_raw = data['signal']['chest']['EDA'].flatten()
    
    # Process Baseline (1) and Stress (2)
    for target_label in [1, 2]:
        indices = np.where(labels == target_label)[0]
        
        for i in range(0, len(indices) - SAMPLES_PER_WINDOW, SAMPLES_PER_STEP):
            start = indices[i]
            end = start + SAMPLES_PER_WINDOW
            
            try:
                w_ecg = ecg_raw[start:end]
                w_eda = eda_raw[start:end]
                
                # ECG Processing
                ecg_signals, _ = nk.ecg_process(w_ecg, sampling_rate=FS)
                hrv = nk.hrv(ecg_signals, sampling_rate=FS)
                
                # EDA Processing
                eda_signals, _ = nk.eda_process(w_eda, sampling_rate=FS)
                eda_f = nk.eda_analyze(eda_signals, sampling_rate=FS)
                
                # Combine Features
                combined = pd.concat([hrv, eda_f], axis=1)
                combined['subject_id'] = subject
                combined['label'] = target_label
                combined['ECG_Rate_Mean'] = ecg_signals["ECG_Rate"].mean()
                
                all_windows_data.append(combined)
            except:
                continue


df = pd.concat(all_windows_data, ignore_index=True)

# remove features with >50% missing values
df = df.dropna(axis=1, thresh=int(0.5 * len(df)))

# FIX: Specify numeric_only=True to avoid TypeError with subject_id strings
df = df.fillna(df.mean(numeric_only=True))

# raw features for visualization before normalization
df_raw = df.copy()

# Per-Subject Z-score Normalization
features_to_norm = [c for c in df.columns if c not in ['subject_id', 'label']]
for subject in SUBJECT_IDS:
    mask = df['subject_id'] == subject
    if mask.any():
        for col in features_to_norm:
            col_mean = df.loc[mask, col].mean()
            col_std = df.loc[mask, col].std()
            if col_std > 0:
                df.loc[mask, col] = (df.loc[mask, col] - col_mean) / col_std
            else:
                df.loc[mask, col] = 0.0

df.to_csv('2wesad_final_normalized_features.csv', index=False)



# --- 4. VISUALIZATIONS ---
sns.set_theme(style="whitegrid")

# Plot 1: Sample Distribution
plt.figure(figsize=(12, 6))
summary_data = df.groupby(['subject_id', 'label']).size().reset_index(name='counts')
summary_data['label'] = summary_data['label'].map({1: 'Baseline', 2: 'Stress'})

sns.barplot(data=summary_data, x='subject_id', y='counts', hue='label')
plt.title('Window Distribution per Subject')
plt.ylabel('Number of 30s Windows')
plt.savefig('data_distribution.png')
plt.close()

# Plot 2: Physiological Response for Subject S2
s2_data = df_raw[df_raw['subject_id'] == 'S2'].copy()
s2_data['label'] = s2_data['label'].map({1: 'Baseline', 2: 'Stress'})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(x='label', y='HRV_RMSSD', data=s2_data, ax=ax1, palette='Set2')
ax1.set_title('Subject S2: HRV RMSSD')
ax1.set_ylabel('RMSSD (ms)')

sns.boxplot(x='label', y='SCR_Peaks_Amplitude_Mean', data=s2_data, ax=ax2, palette='Set2')
ax2.set_title(r'Subject S2: SCR Amplitude ($\mu S$)')
ax2.set_ylabel(r'Amplitude ($\mu S$)')

plt.tight_layout()
plt.savefig('physiology_comparison_s2.png')
plt.close()

print("Process complete. CSV and plots are ready.")

Starting extraction: 30s windows, 5s steps.


Processing Subjects: 100%|██████████| 15/15 [22:29<00:00, 89.96s/it]


Process complete. CSV and plots are ready.
